# ERP anomaly-detection paper: testing tables (Colab)



In [ ]:
from google.colab import files
uploaded  = files.upload()

TRAIN_FILE = "train_001_v1.csv"
TEST_FILE =  "test_001_v1.csv"
TEST2_FILE = "test_002_v1.csv"

In [ ]:
import sys, sklearn
import os, random, warnings, time
import gc

import numpy as np, pandas as pd
import matplotlib.pyplot as plt

from scipy import sparse
from scipy.stats import binomtest, spearmanr

from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import MaxAbsScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220,  "display.max_columns", 60, "display.max_colwidth", 60)

SEED = 42
RESULTS = {}
N_BOOT =  2000
SPLIT_SEEDS = [42, 1, 2, 3,  4]

RUN_LSTM = True
LSTM_EPOCHS =  80
LSTM_CURVE = False

try:
    import tensorflow as tf
    HAVE_TF = True
except Exception:
    HAVE_TF = False


In [ ]:
print("Python", sys.version.split()[0], "| numpy", np.__version__, "| pandas", pd.__version__,
      "| scikit-learn", sklearn.__version__, "| TensorFlow", tf.__version__,
      "| GPU:", bool(tf.config.list_physical_devices("GPU")))

In [ ]:
RUN_LSTM = RUN_LSTM and HAVE_TF

def show(name, df, digits=3):
    RESULTS[name] = df

    print("\n" + name +  "")
    print(df.round(digits).to_string())

In [ ]:
TEXT_COLS = ["module","function_name" ,"table_name","field_changed" ,"action","user_role",
             "trigger_source", "sequence_token",  "old_value","new_value","flow_id", "label"]

def clean(df):

    df = df.copy()
    df["transaction_id"] = df["transaction_id"].astype(str).str.strip()

    df["sequence_no"] = pd.to_numeric(df["sequence_no"], errors="coerce")
    assert df["sequence_no"].notna().all(), "missing sequence_no"
    for c in TEXT_COLS:
        if c not in df.columns: df[c] = "UNKNOWN"
        df[c] = df[c].fillna("UNKNOWN").astype(str).str.strip()
    df["action"] = df["action"].str.upper(); df["label"] = df["label"].str.lower()

    df["flow_id"] = df["flow_id"].str.upper()
    return df.sort_values(["transaction_id","sequence_no"], kind="stable").reset_index(drop=True)

def _cnt_a(v, k): return v.astype(str).str.upper().str.contains(k, regex=False, na=False).sum()
def _cnt_t(v, k): return v.astype(str).str.lower().str.contains(k.lower(), regex=False, na=False).sum()

def _cons(v):
    v = v.astype(str).tolist(); return sum(v[i]==v[i-1] for i in range(1,len(v)))

def _trans(v):
    v = v.astype(str).tolist(); return sum(v[i]!=v[i-1] for i in range(1,len(v)))

def _rep(v):
    c = v.astype(str).value_counts(); return int((c-1).clip(lower=0).sum())

def engineered(df):
    rows = []
    for tid, g in df.groupby("transaction_id", sort=False):
        g = g.sort_values("sequence_no")
        a,f,m,t,fl,r,tr_,tk = (g[c] for c in ["action","function_name","module","table_name",
                                              "field_changed","user_role","trigger_source","sequence_token"])
        L = len(g)

        d = {"transaction_id": tid, "sequence_length": L, "unique_token_count": tk.nunique(),
             "unique_function_count": f.nunique(), "unique_module_count": m.nunique(),
             "unique_table_count": t.nunique(), "unique_field_count": fl.nunique(),
             "unique_action_count": a.nunique(), "unique_role_count": r.nunique(),
             "unique_trigger_count": tr_.nunique(),

             "repeated_token_count": _rep(tk), "repeated_functions_count": _rep(f),
             "consecutive_token_repeats": _cons(tk), "consecutive_function_repeats": _cons(f),
             "token_transition_count": _trans(tk), "function_transition_count": _trans(f),
             "module_transition_count": _trans(m), "table_transition_count": _trans(t),
             "role_transition_count": _trans(r),

             "create_count": _cnt_a(a,"CREATE"), "write_count": _cnt_a(a,"WRITE"),
             "update_count": _cnt_a(a,"UPDATE"), "read_count": _cnt_a(a,"READ"),
             "delete_count": _cnt_a(a,"DELETE"), "confirm_count": _cnt_a(a,"CONFIRM"),
             "validate_count": _cnt_a(a,"VALIDATE"), "post_count": _cnt_a(a,"POST"),
             "state_change_count": _cnt_a(a,"STATE"),

             "purchase_module_count": _cnt_t(m,"purchase"), "stock_module_count": _cnt_t(m,"stock"),
             "account_module_count": _cnt_t(m,"account"),
             "purchase_order_count": _cnt_t(t,"purchase.order"),
             "purchase_line_count": _cnt_t(t,"purchase.order.line"),
             "stock_picking_count": _cnt_t(t,"stock.picking"), "stock_move_count": _cnt_t(t,"stock.move"),

             "account_move_count": _cnt_t(t,"account.move"),
             "quantity_change_count": _cnt_t(fl,"product_qty"), "price_change_count": _cnt_t(fl,"price_unit"),
             "planned_date_change_count": _cnt_t(fl,"date_planned"),
             "vendor_change_count": _cnt_t(fl,"partner_id") + _cnt_t(fl,"vendor"),
             "extreme_value_count": _cnt_t(g["new_value"],"extreme"),
             "high_value_count": _cnt_t(g["new_value"],"high"),
             "state_value_count": _cnt_t(fl,"state")}
        den = max(L,1)

        d["unique_token_ratio"] = d["unique_token_count"]/den
        d["repeated_token_ratio"] = d["repeated_token_count"]/den

        d["write_ratio"] = d["write_count"]/den; d["create_ratio"] = d["create_count"]/den
        d["purchase_ratio"] = d["purchase_module_count"]/den
        d["stock_ratio"] = d["stock_module_count"]/den; d["account_ratio"] = d["account_module_count"]/den
        rows.append(d)
    return pd.DataFrame(rows)

def tx_table(raw):
    df = clean(raw)
    g = df.groupby("transaction_id", sort=False)
    t = g.agg(tokens=("sequence_token", list), scenario=("flow_id","first"),
              label=("label","first")).reset_index()
    t["y"] = (t["label"] != "normal").astype(int)
    if "generation_seed" in df.columns:
        t["gseed"] = g["generation_seed"].first().values
    else:
        t["gseed"] = 0
    return t.merge(engineered(df), on="transaction_id")

tr = tx_table(pd.read_csv(TRAIN_FILE))
te = tx_table(pd.read_csv(TEST_FILE))
ENG = [c for c in tr.columns if c not in ["transaction_id","tokens","scenario","label","y","gseed"]]

assert tr.y.sum() == 0, "training pool must be normal only"
assert len(ENG) == 49, len(ENG)

yte = te.y.values

print("train", len(tr), "| test", len(te), "| anomalies", int(yte.sum()))
print("train generation seeds:", sorted(tr.gseed.unique()))
print(te.scenario.value_counts().sort_index().to_dict())

In [ ]:
#Isolation Forest
#LOF - local outlier factor
#SVM - Support Vector Machine
MODELS = {
    "IF-eng": ("IF","eng",True),
    "IF-count":          ("IF","bag",True),
    "IF-pair":           ("IF","pair",True),
    "LOF-count":         ("LOF","bag",True),
    "LOF-pair":          ("LOF","pair",True),
    "LOF-pair (no OOV)": ("LOF","pair",False),
    "OCSVM-count":       ("OCSVM","bag",True),
    "OCSVM-pair":        ("OCSVM","pair",True),
}
CORE = ["IF-eng","IF-pair","LOF-count","LOF-pair","OCSVM-pair"]

def featurizer(fit_tokens, mode, oov):

    tok = {t:i for i,t in enumerate(sorted({t for s in fit_tokens for t in s}))}

    big = {b:i for i,b in enumerate(sorted({(a,b) for s in fit_tokens for a,b in zip(s,s[1:])}))}

    nt = len(tok)
    def f(tl):
        rows, cols, extra = [], [], []
        for r, s in enumerate(tl):
            o1 = o2 = 0
            for t in s:
                if t in tok: rows.append(r); cols.append(tok[t])
                else: o1 += 1
            if mode == "pair":
                for a,b in zip(s, s[1:]):
                    if (a,b) in big: rows.append(r); cols.append(nt+big[(a,b)])
                    else: o2 += 1
            extra.append([len(s), o1, o2] if mode=="pair" else [len(s), o1])

        n = nt + (len(big) if mode=="pair" else 0)
        X = sparse.csr_matrix((np.ones(len(rows)), (rows,cols)), shape=(len(tl), n))
        E = np.array(extra, float)

        if not oov: E = E[:, :1]
        return sparse.hstack([X, sparse.csr_matrix(E)]).tocsr()
    return f

def sk_scores(mid, fit, cal, tests, seed=SEED, k=20, eng_cols=None):
    fam, feat, oov = MODELS[mid]
    if feat == "eng":
        cols = eng_cols or ENG
        mk = lambda d: d[cols].replace([np.inf,-np.inf], np.nan).fillna(0).values.astype(float)
        Xf, Xc, Xts = mk(fit), mk(cal), [mk(d) for d in tests]
    else:
        fz = featurizer(fit.tokens, "pair" if feat=="pair" else "bag", oov)

        Xf, Xc, Xts = (
          fz(fit.tokens),
          fz(cal.tokens),
          [fz(d.tokens) for d in tests]
        )


        Xf, Xc, Xts = fz(fit.tokens), fz(cal.tokens), [fz(d.tokens) for d in tests]
        sc = MaxAbsScaler().fit(Xf)
        Xf, Xc, Xts = sc.transform(Xf), sc.transform(Xc), [sc.transform(x) for x in Xts]
    if fam == "IF":
        m = IsolationForest(n_estimators=500, max_samples="auto", contamination="auto",
                            random_state=seed, n_jobs=-1).fit(Xf)

    elif fam == "LOF":
        m = LocalOutlierFactor(n_neighbors=k, novelty=True, n_jobs=-1).fit(Xf)
    else:
        m = OneClassSVM(kernel="rbf", gamma="scale", nu=0.05).fit(Xf)
    f = lambda X: -m.score_samples(X)
    return f(Xc), [f(X) for X in Xts]


def conf(y, s, thr):
    pred = (s > thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0,1]).ravel()
    P = tp/(tp+fp) if tp+fp else 0.0; R = tp/(tp+fn)
    F = 2*P*R/(P+R) if P+R else 0.0

    return dict(acc=(tp+tn)/len(y), prec=P, rec=R, f1=F, fpr=fp/(fp+tn), tp=tp, fp=fp, tn=tn, fn=fn), pred

def summarize(y, s_cal, s_te, pcts=(95,99)):
    out = dict(auc=roc_auc_score(y, s_te), ap=average_precision_score(y, s_te))
    for p in pcts:
        m, _ = conf(y, s_te, np.percentile(s_cal, p))
        out.update({f"{k}@{p}": m[k] for k in ("prec","rec","f1","fpr")})
    return out

def split(seed=SEED):
    return train_test_split(np.arange(len(tr)), test_size=0.2, random_state=seed, shuffle=True)

fit_i, cal_i = split(SEED)
FIT, CAL = tr.iloc[fit_i], tr.iloc[cal_i]

In [ ]:
#IF-eng
sv, (st,) = sk_scores("IF-eng", FIT, CAL, [te])
m, _ = conf(yte, st, np.quantile(sv, 0.95))
print("IF-eng@95: acc %.4f prec %.4f rec %.4f f1 %.4f  AUC %.4f AP %.4f" %
      (m["acc"], m["prec"], m["rec"], m["f1"], roc_auc_score(yte, st), average_precision_score(yte, st)))

In [ ]:
#LSTM
LSTM_CACHE = {}
if RUN_LSTM:
    def l_encode(tl, vocab, L):
        X = np.zeros((len(tl), L), dtype="int32")
        for i, s in enumerate(tl):
            ids = [vocab.get(t, 1) for t in s][:L]          # 0 = PAD, 1 = OOV
            X[i, :len(ids)] = ids
        return X

    def l_build(V, L):
        K = tf.keras.layers
        inp = tf.keras.Input(shape=(L,), dtype="int32")
        x = K.Embedding(V, 32, mask_zero=True)(inp)
        x = K.LSTM(64, dropout=0.2)(x)
        x = K.RepeatVector(L)(x)
        x = K.LSTM(64, return_sequences=True, dropout=0.2)(x)
        out = K.TimeDistributed(K.Dense(V, activation="softmax"))(x)
        m = tf.keras.Model(inp, out)

        m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=tf.keras.losses.SparseCategoricalCrossentropy())
        return m

    def lstm_fit(fit_tokens, cal_tokens, seed, epochs=None, maxlen=None):
        epochs = epochs or LSTM_EPOCHS
        tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(seed)
        vocab = {t: i for i, t in enumerate(sorted({t for s in fit_tokens for t in s}), start=2)}
        L = maxlen or max(len(s) for s in list(fit_tokens) + list(cal_tokens))
        Xf, Xc = l_encode(fit_tokens, vocab, L), l_encode(cal_tokens, vocab, L)

        model = l_build(len(vocab) + 2, L)

        cb = [tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, min_delta=1e-4, restore_best_weights=True),
              tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-5)]

        model.fit(Xf, Xf, sample_weight=(Xf != 0).astype("float32"),
                  validation_data=(Xc, Xc, (Xc != 0).astype("float32")),
                  epochs=epochs, batch_size=32, shuffle=True, callbacks=cb, verbose=0)
        return dict(model=model, vocab=vocab, L=L)

    def lstm_score(res, tl, agg="mean"):
        X = l_encode(tl, res["vocab"], res["L"])
        prob = res["model"].predict(X, batch_size=128, verbose=0)
        p = prob[np.arange(len(X))[:, None], np.arange(X.shape[1])[None, :], X]
        nll = -np.log(np.clip(p, 1e-8, 1.0)); mask = X != 0; n = np.maximum(mask.sum(1), 1)
        if agg == "mean": return (nll * mask).sum(1) / n
        if agg == "max":  return np.where(mask, nll, -1.0).max(1)
        if agg == "top3":
            top = np.sort(np.where(mask, nll, 0.0), axis=1)[:, ::-1][:, :3]
            return top.sum(1) / np.minimum(n, 3)
        raise ValueError(agg)

    def get_lstm(key, fit_df, cal_df, seed, maxlen=None):
        if key not in LSTM_CACHE:
            t0 = time.time()
            LSTM_CACHE[key] = lstm_fit(fit_df.tokens.tolist(), cal_df.tokens.tolist(), seed, maxlen=maxlen)
            print(f"  LSTM fit {key}: {time.time()-t0:.0f}s")
        return LSTM_CACHE[key]

    def lstm_scores(key, fit_df, cal_df, tests, seed, agg="mean"):
        res = get_lstm(key, fit_df, cal_df, seed)
        return lstm_score(res, cal_df.tokens.tolist(), agg), [lstm_score(res, d.tokens.tolist(), agg) for d in tests]

In [ ]:
S = {}
#Models (IF, LOF, SVM) RUN
for mid in MODELS:
    sv, (st,) = sk_scores(mid, FIT, CAL, [te]); S[mid] = (sv, st)

In [ ]:
#LSTM RUN
if RUN_LSTM:
    sv, (st,) = lstm_scores("split42", FIT, CAL, [te], SEED); S["LSTM-AE"] = (sv, st)


ORDER = (["LSTM-AE"] if RUN_LSTM else []) + list(MODELS)
SCEN = sorted(te.scenario.unique())

def boot_idx(y, rng):
    i0, i1 = np.where(y == 0)[0], np.where(y == 1)[0]
    return np.concatenate([rng.choice(i0, len(i0)), rng.choice(i1, len(i1))])

def boot_ci(y, s, fn=roc_auc_score, n=N_BOOT, seed=0):
    rng = np.random.default_rng(seed)
    v = [fn(y[i], s[i]) for i in (boot_idx(y, rng) for _ in range(n))]
    return np.percentile(v, [2.5, 97.5])

rows = []
for mid in ORDER:
    sv, st = S[mid]; r = dict(model=mid, **summarize(yte, sv, st))
    lo, hi = boot_ci(yte, st); r["auc_95ci"] = f"[{lo:.3f}, {hi:.3f}]"
    rows.append(r)

cols = ["model","auc","auc_95ci","ap","prec@95","rec@95","f1@95","fpr@95","prec@99","rec@99","f1@99","fpr@99"]

show("TABLE 1  Main Comparison ( default split, test set N=300; thresholds = 95th / 99th percentile of normal calibration scores)",
     pd.DataFrame(rows)[cols].set_index("model"))

In [ ]:
rows = []
for mid in ORDER:
    sv, st = S[mid]
    pct = 95 if mid.startswith("IF-eng") else 99          # IF 95
    pred = st > np.percentile(sv, pct)

    tp = int((pred & (yte == 1)).sum());    fp = int((pred & (yte == 0)).sum())
    tn = int((~pred & (yte == 0)).sum()); fn = int((~pred & (yte == 1)).sum())

    rows.append(dict(model=mid, pct=pct, TP=tp, FP=fp, TN=tn, FN=fn, acc=round(100 * (tp + tn) / len(yte), 2)))
show("TABLE 1b  confusion counts and accuracy (same split, test set 1)", pd.DataFrame(rows).set_index("model"))

In [ ]:
# TABLE 2 - per-scenario flagged counts
def flagged_table(pct):
    out = {}
    for mid in ORDER:
        sv, st = S[mid]; pred = (st > np.percentile(sv, pct)).astype(int)
        out[mid] = pd.Series(pred).groupby(te.scenario.values).sum()

    n = pd.Series(1, index=range(len(te))).groupby(te.scenario.values).sum()
    T = pd.DataFrame(out).T[SCEN]
    T.loc["(n per scenario)"] = n[SCEN]
    return T

show("TABLE 2a  flagged transactions per scenario, 95th-percentile threshold", flagged_table(95), 0)

show("TABLE 2b  flagged transactions per scenario , 99th percentile threshold", flagged_table(99), 0)

In [ ]:
#  TABLE 3: paired comparisons

def paired(a, b, n=N_BOOT, seed=1):
    (sva, sta), (svb, stb) = S[a], S[b]
    rng = np.random.default_rng(seed)
    d = []

    for _ in range(n):
        i = boot_idx(yte, rng); d.append(roc_auc_score(yte[i], sta[i]) - roc_auc_score(yte[i], stb[i]))
    d = np.array(d); lo, hi = np.percentile(d, [2.5, 97.5])
    pa = (sta > np.percentile(sva, 99)); pb = (stb > np.percentile(svb, 99))

    ca, cb = (pa == yte), (pb == yte)
    b01, b10 = int((ca & ~cb).sum()), int((~ca & cb).sum())

    p = binomtest(b01, b01 + b10, 0.5).pvalue if b01 + b10 else 1.0
    return dict(A=a, B=b, dAUC=roc_auc_score(yte, sta) - roc_auc_score(yte, stb),
                ci95=f"[{lo:.3f}, {hi:.3f}]", excludes_0=bool(lo > 0 or hi < 0),
                A_right_B_wrong=b01, B_right_A_wrong=b10, mcnemar_p=p)


pairs = [("LOF-pair","IF-eng"), ("LOF-pair","OCSVM-pair"), ("LOF-pair","LOF-count"), ("IF-eng","IF-pair")]

if RUN_LSTM: pairs = [("LSTM-AE","IF-eng"), ("LSTM-AE","LOF-pair"), ("LSTM-AE","OCSVM-pair")] + pairs


show("TABLE 3  paired stratified bootstrap of the AUC difference and McNemar test at the 99th percentile", pd.DataFrame([paired(*p) for p in pairs]).set_index(["A","B"]))

In [ ]:
# TABLE 4 - repeated calibration splits
def run_split(seed, ids):
    fi, ci = split(seed); F, C = tr.iloc[fi], tr.iloc[ci]; out = {}

    for mid in ids:
        if mid == "LSTM-AE": sv, (st,) = lstm_scores(f"split{seed}", F, C, [te], seed)
        else:                sv, (st,) = sk_scores(mid, F, C, [te], seed=seed)
        r = summarize(yte, sv, st); pred = (st > np.percentile(sv, 99)).astype(int)
        r["FP_N@99"] = int(pred[(te.scenario.str[0] == "N").values].sum())
        r["FP_G@99"] = int(pred[(te.scenario.str[0] == "G").values].sum())
        out[mid] = r
    return out

ids4 = (["LSTM-AE"] if RUN_LSTM else []) + CORE
runs = [run_split(s, ids4) for s in SPLIT_SEEDS]
rows = []
for mid in ids4:
    d = pd.DataFrame([r[mid] for r in runs])
    row = {"model": mid, "runs": len(d)}
    for c in ["auc","ap","f1@95","f1@99","fpr@95","fpr@99","FP_G@99"]:
        row[c] = f"{d[c].mean():.3f} ± {d[c].std(ddof=1):.3f}"
    rows.append(row)

show("TABLE 4  mean ± SD over %d calibration splits " % len(SPLIT_SEEDS),
     pd.DataFrame(rows).set_index("model"))

In [ ]:
# TABLE 5: leave-one-generation-seed-out calibration
gs = sorted(tr.gseed.unique())
if len(gs) < 2:
    print("train file has no generation_seed variation; skipping Table 5")
else:
    ids5 = (["LSTM-AE"] if RUN_LSTM else []) + CORE; res5 = {m: [] for m in ids5}
    for g in gs:
        F, C = tr[tr.gseed != g], tr[tr.gseed == g]
        for mid in ids5:
            if mid == "LSTM-AE": sv, (st,) = lstm_scores(f"loso{g}", F, C, [te], SEED)
            else:                sv, (st,) = sk_scores(mid, F, C, [te])
            r = summarize(yte, sv, st); pred = (st > np.percentile(sv, 99)).astype(int)
            r["FP_G@99"] = int(pred[(te.scenario.str[0] == "G").values].sum()); res5[mid].append(r)
    rows = []
    for mid in ids5:
        d = pd.DataFrame(res5[mid]); row = {"model": mid, "folds": len(d)}
        for c in ["auc","f1@95","f1@99","fpr@95","fpr@99","FP_G@99"]:
            row[c] = f"{d[c].mean():.3f} ± {d[c].std(ddof=1):.3f}"
        rows.append(row)
    show("TABLE 5  leave-one-generation-seed-out", pd.DataFrame(rows).set_index("model"))

In [ ]:
#  TABLE 6: training-size curve
sizes = [50, 100, 200, 400, 800, 1200]
rng = np.random.default_rng(0); rows = []
for n in sizes:
    for rep in range(3):
        sub = FIT.iloc[rng.permutation(len(FIT))[:n]]
        for mid in CORE:
            sv, (st,) = sk_scores(mid, sub, CAL, [te])
            rows.append(dict(n=n, model=mid, rep=rep, auc=roc_auc_score(yte, st),
                             f1_99=summarize(yte, sv, st)["f1@99"]))
if RUN_LSTM and LSTM_CURVE:
    for n in [100, 400, 1200]:
        sub = FIT.iloc[rng.permutation(len(FIT))[:n]]
        sv, (st,) = lstm_scores(f"curve{n}", sub, CAL, [te], SEED)
        rows.append(dict(n=n, model="LSTM-AE", rep=0, auc=roc_auc_score(yte, st), f1_99=summarize(yte, sv, st)["f1@99"]))
d = pd.DataFrame(rows)
show("TABLE 6a  AUC vs number of normal training transactions", d.pivot_table(index="model", columns="n", values="auc"))
show("TABLE 6b  F1 at 99th percentile vs training size", d.pivot_table(index="model", columns="n", values="f1_99"))

In [ ]:
#E7
SIZES   = [50, 100, 200, 400, 800, 1200]
REPS    = 3
CKPT    = "lstm_curve_rows.csv"
SK_IDS  = CORE + ["LOF-pair (no OOV)"]
MODELS_ORDER = ["LSTM-AE"] + SK_IDS

L0  = max(len(s) for s in tr.tokens)
is_N = (te.scenario.str[0] == "N").values
is_G = (te.scenario.str[0] == "G").values

rows = pd.read_csv(CKPT).to_dict("records") if os.path.exists(CKPT) else []
have = {(int(r["n"]), int(r["rep"]), r["model"]) for r in rows}
if rows:
    print(f"Resuming: {len(rows)} rows already in {CKPT}")

def row_from(n, rep, model, sv, st):
    thr  = np.percentile(sv, 99)
    pred = st > thr
    s = summarize(yte, sv, st)
    return dict(n=n, rep=rep, model=model, auc=s["auc"], f1_99=s["f1@99"],
                fpr_99=s["fpr@99"], FP_N_99=int(pred[is_N].sum()), FP_G_99=int(pred[is_G].sum()))

rng = np.random.default_rng(0)
t_start = time.time()
for n in SIZES:
    for rep in range(REPS):
        perm = rng.permutation(len(FIT))
        sub  = FIT.iloc[perm[:n]]

        for mid in SK_IDS:
            if (n, rep, mid) in have: continue
            sv, (st,) = sk_scores(mid, sub, CAL, [te])
            rows.append(row_from(n, rep, mid, sv, st)); have.add((n, rep, mid))

        if (n, rep, "LSTM-AE") not in have:
            t0  = time.time()
            res = lstm_fit(sub.tokens.tolist(), CAL.tokens.tolist(), SEED + rep, maxlen=L0)
            cal_t, te_t = CAL.tokens.tolist(), te.tokens.tolist()
            sv, st   = lstm_score(res, cal_t), lstm_score(res, te_t)
            svm, stm = lstm_score(res, cal_t, "max"), lstm_score(res, te_t, "max")
            r = row_from(n, rep, "LSTM-AE", sv, st)
            r["auc_max"] = float(roc_auc_score(yte, stm))
            rows.append(r); have.add((n, rep, "LSTM-AE"))
            del res; gc.collect()
            if HAVE_TF: tf.keras.backend.clear_session()
            print(f"n={n:4d} rep={rep}  LSTM AUC={r['auc']:.3f}  max-error AUC={r['auc_max']:.3f}  "
                  f"FP_G={r['FP_G_99']:2d}/75  ({time.time()-t0:.0f}s, total {(time.time()-t_start)/60:.1f} min)")

        pd.DataFrame(rows).to_csv(CKPT, index=False)

d = pd.DataFrame(rows)
d.to_csv("lstm_curve_all_rows.csv", index=False)

def ms(x): return f"{x.mean():.3f} ± {x.std(ddof=1):.3f}"
def tbl(col, fmt=ms):
    t = d.groupby(["model", "n"])[col].agg(fmt).unstack("n")
    return t.reindex([m for m in MODELS_ORDER if m in t.index])

show("TABLE 6a+  ROC-AUC vs number of normal training transactions (mean ± SD, 3 random subsets)", tbl("auc"))
show("TABLE 6b+  F1 at 99th percentile vs training size", tbl("f1_99"))
show("TABLE 6c   False positives on the 75 unseen legitimate workflows (G) at 99th percentile vs training size", tbl("FP_G_99"))
show("TABLE 6d   LSTM only: ROC-AUC with MAX token error as the score vs training size",
     d[d.model == "LSTM-AE"].groupby("n")["auc_max"].agg(ms).to_frame("LSTM-AE (max error)").T)

fig, ax = plt.subplots(figsize=(6.2, 3.8))
for m in MODELS_ORDER:
    g = d[d.model == m].groupby("n")["auc"].agg(["mean", "std"]).reindex(SIZES)
    ax.errorbar(SIZES, g["mean"], yerr=g["std"], marker="o", capsize=2,
                lw=2.2 if m == "LSTM-AE" else 1.2, label=m)
ax.set_xscale("log"); ax.set_xticks(SIZES); ax.set_xticklabels(SIZES)
ax.set_xlabel("Normal training transactions"); ax.set_ylabel("ROC-AUC")
ax.legend(fontsize=7, frameon=False); fig.tight_layout()
os.makedirs("figures", exist_ok=True)
fig.savefig("figures/fig6b_training_size_with_lstm.png", dpi=300)
plt.show()

print("\nCheck: LSTM-AE at n=1200 (mean over reps):",
      round(d[(d.model == "LSTM-AE") & (d.n == 1200)]["auc"].mean(), 3))

In [ ]:
#  TB 07: out-of-vocabulary (OOV)
vocab_fit = {t for s in FIT.tokens for t in s}
pairs_fit = {(a, b) for s in FIT.tokens for a, b in zip(s, s[1:])}
te["has_oov"] = [any(t not in vocab_fit for t in s) for s in te.tokens]
te["has_new_pair"] = [any(p not in pairs_fit for p in zip(s, s[1:])) for s in te.tokens]
prev = te.groupby("scenario")[["has_oov","has_new_pair"]].mean()
prev["unseen_tokens"] = te.groupby("scenario").tokens.apply(lambda x: sorted({t for s in x for t in s if t not in vocab_fit})[:4])
show("TABLE 7a  share of test transactions containing an unseen token / unseen token pair", prev)

keep = ~te.has_oov.values
rows = []
for mid in ORDER:
    sv, st = S[mid]
    r = dict(model=mid, auc_all=roc_auc_score(yte, st), n_all=len(te))
    if keep.sum() and len(set(yte[keep])) == 2:
        r.update(auc_oov_free=roc_auc_score(yte[keep], st[keep]), n_oov_free=int(keep.sum()),
                 anomalies_oov_free=int(yte[keep].sum()))
    rows.append(r)
show("TABLE 7b  AUC on all test transactions vs only transactions with NO unseen token", pd.DataFrame(rows).set_index("model"))

# detection rates in A1/A2 with and without OOV counts as features
rows = []
for mid in ["LOF-pair", "LOF-pair (no OOV)"]:
    sv, st = S[mid]; pred = (st > np.percentile(sv, 99)).astype(int)
    rows.append(pd.Series(pred).groupby(te.scenario.values).sum().rename(mid))
show("TABLE 7c  LOF flagged per scenario at 99th: with vs without explicit OOV-count features", pd.DataFrame(rows)[SCEN], 0)


print('\n')
a1 = te[te.scenario=="A1"]
print(a1.tokens.map(lambda s: sum(t not in vocab_fit for t in s)).value_counts().sort_index())
a2 = te[te.scenario=="A2"]
print(a2.tokens.map(lambda s: sum(t not in vocab_fit for t in s)).value_counts().sort_index())
n5 = te[te.scenario=="N5"]
print(n5.tokens.map(lambda s: sum(t not in vocab_fit for t in s)).value_counts().sort_index())


In [ ]:
total_tokens = te.groupby("scenario").tokens.apply(
    lambda x: sum(len(s) for s in x)
)

print(total_tokens)

print('\n')
unseen = {t for s in te.tokens for t in s if t not in vocab_fit}

print("Total distinct unseen tokens:", len(unseen))
print("Unseen token identities:")
for t in sorted(unseen):
    print(t)

In [ ]:
#  TABLE 8: Event Order - Shuffle
rng = np.random.default_rng(7); NREP = 5
shuf = []
for _ in range(NREP):
    d = te.copy(); d["tokens"] = [list(rng.permutation(np.array(s, dtype=object))) for s in te.tokens]; shuf.append(d)
ids8 = ["LOF-count", "LOF-pair", "OCSVM-pair", "IF-pair"]
rows = []

for mid in ids8:
    sv, outs = sk_scores(mid, FIT, CAL, [te] + shuf)
    a = [roc_auc_score(yte, o) for o in outs]
    rows.append(dict(model=mid, auc_original=a[0], auc_shuffled=f"{np.mean(a[1:]):.3f} ± {np.std(a[1:], ddof=1):.3f}",
                     drop=a[0] - np.mean(a[1:])))
if RUN_LSTM:
    res = get_lstm("split42", FIT, CAL, SEED)
    a = [roc_auc_score(yte, lstm_score(res, d.tokens.tolist())) for d in [te] + shuf]
    rows.insert(0, dict(model="LSTM-AE", auc_original=a[0], auc_shuffled=f"{np.mean(a[1:]):.3f} ± {np.std(a[1:], ddof=1):.3f}", drop=a[0] - np.mean(a[1:])))

show("TABLE 8  AUC with original vs randomly shuffled event order (%d shuffles)." % NREP,
     pd.DataFrame(rows).set_index("model"))

In [ ]:
# TABLE 9: A2 - token-level scoring (LSTM only)
if RUN_LSTM:
    res = get_lstm("split42", FIT, CAL, SEED); rows = []
    for agg in ["mean", "max", "top3"]:
        sv = lstm_score(res, CAL.tokens.tolist(), agg); st = lstm_score(res, te.tokens.tolist(), agg)

        for p in (95, 99):
            pred = (st > np.percentile(sv, p)).astype(int)
            fl = pd.Series(pred).groupby(te.scenario.values).sum()
            m, _ = conf(yte, st, np.percentile(sv, p))
            rows.append(dict(aggregation=agg, pct=p, auc=roc_auc_score(yte, st), f1=m["f1"], fpr=m["fpr"],
                             **{s: int(fl[s]) for s in SCEN}))

    show("TABLE 9  LSTM sequence score: mean vs max vs mean-of-top-3 token errors", pd.DataFrame(rows).set_index(["aggregation","pct"]))

else:
    print("")

In [ ]:
# ------ TABLE 10: Isolation-Forest feature-group ablation --------
GROUPS = {
  "length/diversity": ["sequence_length","unique_token_count","unique_function_count","unique_module_count","unique_table_count",
                       "unique_field_count","unique_action_count","unique_role_count","unique_trigger_count"],

  "repetition/transition": ["repeated_token_count","repeated_functions_count","consecutive_token_repeats","consecutive_function_repeats",
                       "token_transition_count","function_transition_count","module_transition_count","table_transition_count","role_transition_count"],

  "action counts": ["create_count","write_count","update_count","read_count","delete_count","confirm_count","validate_count","post_count","state_change_count"],

  "module/table counts": ["purchase_module_count","stock_module_count","account_module_count","purchase_order_count","purchase_line_count",
                       "stock_picking_count","stock_move_count","account_move_count"],

  "field/value counts": ["quantity_change_count","price_change_count","planned_date_change_count","vendor_change_count",
                       "extreme_value_count","high_value_count","state_value_count"],

  "ratios": ["unique_token_ratio","repeated_token_ratio","write_ratio","create_ratio","purchase_ratio","stock_ratio","account_ratio"],
}
assert sorted(sum(GROUPS.values(), [])) == sorted(ENG), "49 features exactly"

def if_eval(cols):
    a, f = [], []
    for sd in (42, 1, 2):
        sv, (st,) = sk_scores("IF-eng", FIT, CAL, [te], seed=sd, eng_cols=cols)
        a.append(roc_auc_score(yte, st)); f.append(summarize(yte, sv, st)["f1@95"])
    return np.mean(a), np.mean(f)
rows = []; a, f = if_eval(ENG); rows.append(dict(setting="all 49 features", n_features=49, auc=a, f1_95=f))

for g, cols in GROUPS.items():
    a, f = if_eval([c for c in ENG if c not in cols]); rows.append(dict(setting=f"drop: {g}", n_features=49-len(cols), auc=a, f1_95=f))
for g, cols in GROUPS.items():
    a, f = if_eval(cols); rows.append(dict(setting=f"only: {g}", n_features=len(cols), auc=a, f1_95=f))

show("TABLE 10  IF feature-group ablation (mean of 3 Forests)", pd.DataFrame(rows).set_index("setting"))

In [ ]:
# TABLE 11: LOF sensitivity to k and threshold percentile
rows = []
for k in [5, 10, 20, 50, 100]:

    sv, (st,) = sk_scores("LOF-pair", FIT, CAL, [te], k=k); r = dict(k=k, auc=roc_auc_score(yte, st))

    for p in (95, 97.5, 99, 99.5):
        m, pred = conf(yte, st, np.percentile(sv, p))
        r[f"f1@{p}"] = m["f1"]; r[f"fpr@{p}"] = m["fpr"]
        r[f"FP_G@{p}"] = int(pred[(te.scenario.str[0] == "G").values].sum())
    rows.append(r)
show("TABLE 11  LOF-pair", pd.DataFrame(rows).set_index("k"))

In [ ]:
# TABLE 12: G5
rows = []
cal_sorted = {m: np.sort(S[m][0]) for m in ORDER}
for sc in SCEN:
    d = te[te.scenario == sc]; idx = d.index.values
    r = dict(scenario=sc, mean_length=d.sequence_length.mean(),
             share_longer_than_train_max=float((d.sequence_length > FIT.sequence_length.max()).mean()),
             unseen_token_rate=np.mean([np.mean([t not in vocab_fit for t in s]) for s in d.tokens]),
             unseen_pair_rate=np.mean([np.mean([p not in pairs_fit for p in zip(s, s[1:])] or [0]) for s in d.tokens]))
    for m in ["IF-eng", "LOF-pair"] + (["LSTM-AE"] if RUN_LSTM else []):
        r[f"{m} pct-rank"] = np.mean(np.searchsorted(cal_sorted[m], S[m][1][idx]) / len(cal_sorted[m]))
    rows.append(r)
show("TABLE 12a  scenario profile",
     pd.DataFrame(rows).set_index("scenario"))
rows = []
nm = te.y.values == 0
for m in ORDER:
    rho, _ = spearmanr(te.sequence_length[nm], S[m][1][nm]); rows.append(dict(model=m, spearman_length_vs_score_normal_and_G=rho))
show("TABLE 12b  correlation of score with transaction length on legitimate (N+G) test transactions", pd.DataFrame(rows).set_index("model"))

In [ ]:
# == E3: A2 / A4 / A5 / G1 / G5 diagnostics


FEAT = ["sequence_length","write_count","planned_date_change_count","repeated_functions_count",
        "consecutive_function_repeats","repeated_token_count","consecutive_token_repeats"]

sv_if, st_if = S["IF-eng"]; thr_if = np.percentile(sv_if, 95)
dI = te.assign(if_score=st_if, if_flag=st_if > thr_if)

rows = []

for sc in ["A2", "A4", "A5", "G1", "G5"]:

    x = dI[dI.scenario == sc]
    for flag, g in x.groupby("if_flag"):
        rows.append(dict(scenario=sc, IF_flagged=bool(flag), n=len(g), score_min=g.if_score.min(), score_max=g.if_score.max(),
                         **{c: g[c].mean() for c in FEAT}))

show("TABLE 13a  E3 Isolation Forest: A4 / A5 / G1 (and A2, G5)",
     pd.DataFrame(rows).set_index(["scenario", "IF_flagged"]), 4)

x = dI[dI.scenario == "A4"]; rows = []
for c in FEAT + ["unique_token_count"]:
    rho = spearmanr(x[c], x.if_score)[0] if x[c].nunique() > 1 else np.nan
    rows.append(dict(feature=c, distinct_values_in_A4=x[c].nunique(), spearman_with_IF_score=rho))
show("TABLE 13b  A4: Spearman correlation of each feature with the IF score)",
     pd.DataFrame(rows).set_index("feature"))

if RUN_LSTM:

    res = get_lstm("split42", FIT, CAL, SEED); sv, st = S["LSTM-AE"]; thr = np.percentile(sv, 99)
    dL = te.assign(lstm=st, lstm_flag=st > thr); rows = []
    for sc in ["A2", "A4", "A5", "G1", "G5"]:
        g = dL[dL.scenario == sc]
        rows.append(dict(scenario=sc, n=len(g), flagged=int(g.lstm_flag.sum()), score_min=g.lstm.min(), score_mean=g.lstm.mean(),
                         score_max=g.lstm.max(), len_min=g.sequence_length.min(), len_max=g.sequence_length.max()))
    show("TABLE 13c  E3 LSTM: A2 / A4 / A5 / G1 / G5 (99th-percentile threshold = %.4f)" % thr, pd.DataFrame(rows).set_index("scenario"), 4)
    g = dL[dL.scenario == "A2"]
    show("TABLE 13d  A2 detected vs missed by the  lstm",
         g.groupby("lstm_flag").agg(n=("lstm","size"), mean_length=("sequence_length","mean"), min_len=("sequence_length","min"),
                                    score_min=("lstm","min"), score_mean=("lstm","mean"), score_max=("lstm","max")), 4)

    TOKEN_IDS = []        #["P00193", "P00186"]
    ids = TOKEN_IDS or (list(g[~g.lstm_flag].sort_values("lstm").transaction_id.head(2)) +
                        list(g[g.lstm_flag].sort_values("lstm").transaction_id.head(2)))
    X = l_encode(te.tokens.tolist(), res["vocab"], res["L"])
    prob = res["model"].predict(X, batch_size=128, verbose=0)
    rows = []

    for tid in ids:
        i = int(np.where(te.transaction_id.values == tid)[0][0]); toks = te.tokens.iloc[i][:res["L"]]
        for pos, t in enumerate(toks):
            rows.append(dict(transaction_id=tid, flagged=bool(dL.lstm_flag.iloc[i]), position=pos + 1, token=t,
                             token_error=-np.log(np.clip(prob[i, pos, X[i, pos]], 1e-8, 1))))
    tok = pd.DataFrame(rows)
    show("TABLE 13e  A2 token-level errors: the 5 highest-error events in 2 missed and 2 detected A2 transactions",
         tok.sort_values(["transaction_id", "token_error"], ascending=[True, False]).groupby("transaction_id").head(5).set_index("transaction_id"), 4)
else:
    print("Tables 13c-13e")

In [ ]:
#  MAXIMUM SEQUENCE LENGTH
L0 = max(len(s) for s in list(FIT.tokens) + list(CAL.tokens))
d = te.assign(over=(te.sequence_length - L0).clip(lower=0))
t = d.groupby("scenario").agg(mean_length=("sequence_length","mean"), max_length=("sequence_length","max"),
                              share_over=("over", lambda v: (v > 0).mean()), events_truncated=("over","sum"))
show("TABLE 14a  longest normal training trace = %d events" % L0, t, 3)

te_cut = te.copy(); te_cut["tokens"] = [s[:L0] for s in te.tokens]
rows = []
for mid in ["LOF-count", "LOF-pair", "OCSVM-pair", "IF-pair"]:
    sv, (full, cut) = sk_scores(mid, FIT, CAL, [te, te_cut])
    for name, s_ in [("full traces", full), ("cut at %d" % L0, cut)]:
        pred = (s_ > np.percentile(sv, 99)).astype(int)
        rows.append(dict(model=mid, test_input=name, auc=roc_auc_score(yte, s_),
                         G5_flagged=int(pred[(te.scenario == "G5").values].sum()),
                         FP_G=int(pred[(te.scenario.str[0] == "G").values].sum()), FP_N=int(pred[(te.scenario.str[0] == "N").values].sum())))
show("TABLE 14b  control: truncating the test traces changes the length-free", pd.DataFrame(rows).set_index(["model","test_input"]))

if RUN_LSTM:
    rows = []
    for ml in sorted({L0, 33, 40}):
        key = "split42" if ml == L0 else f"maxlen{ml}"
        res = get_lstm(key, FIT, CAL, SEED, maxlen=None if ml == L0 else ml)
        sv = lstm_score(res, CAL.tokens.tolist()); st = lstm_score(res, te.tokens.tolist())
        m, pred = conf(yte, st, np.percentile(sv, 99)); sc = te.scenario.values
        rows.append(dict(max_len=ml, auc=roc_auc_score(yte, st), acc=m["acc"], prec=m["prec"], rec=m["rec"], f1=m["f1"],
                         FP_N=int(pred[te.scenario.str[0].values == "N"].sum()), FP_G=int(pred[te.scenario.str[0].values == "G"].sum()),
                         G1_FP=int(pred[sc == "G1"].sum()), G5_FP=int(pred[sc == "G5"].sum()), A2_detected=int(pred[sc == "A2"].sum()),
                         threshold=np.percentile(sv, 99)))
    show("TABLE 14c  LSTM retrained with different maximum lengths ",
         pd.DataFrame(rows).set_index("max_len"), 4)
else:
    print("Table 14c")

In [ ]:
#  E4: trace-representation ablation
def rep_tok(tok, name):
    tok = str(tok); p = tok.split(":")
    if name == "R1 full":                      return tok
    if name == "R2 no value":                  return tok.rsplit(":", 1)[0] if ":" in tok else tok
    if name == "R3 no role":                   return tok.split(":", 1)[1]
    if name == "R4 structural":                return ":".join(p[1:-1])
    if name == "R2* no value (field-aware)":   return ":".join(p[:-1]) if len(p) == 6 else tok
    if name == "R4* structural (field-aware)": return ":".join(p[1:-1]) if len(p) == 6 else ":".join(p[1:])
    raise ValueError(name)

REPS_MAIN = ["R1 full", "R2 no value", "R3 no role", "R4 structural"]
REPS_EXTRA = ["R2* no value (field-aware)", "R4* structural (field-aware)"]
E4_SEEDS = SPLIT_SEEDS[:3]
n4 = tr.tokens.explode().str.count(":").value_counts().to_dict(); print("colons per token in train:", n4)

def with_rep(df, name):
    d = df.copy(); d["tokens"] = [[rep_tok(t, name) for t in s] for s in df.tokens]; return d

def row99(y, sv, st, sc):
    m, pred = conf(y, st, np.percentile(sv, 99))
    return dict(auc=roc_auc_score(y, st), acc=m["acc"], prec=m["prec"], rec=m["rec"], f1=m["f1"], tp=m["tp"], fn=m["fn"],
                FP_N=int(pred[sc == "N"].sum()), FP_G=int(pred[sc == "G"].sum()))
scl = te.scenario.str[0].values
allrows = []

for seed in E4_SEEDS:
    fi, ci = split(seed)
    for rep in REPS_MAIN + REPS_EXTRA:
        F, C, T = (with_rep(tr.iloc[fi], rep), with_rep(tr.iloc[ci], rep), with_rep(te, rep))
        for mid in ["LOF-count", "LOF-pair"]:
            sv, (st,) = sk_scores(mid, F, C, [T], seed=seed)
            allrows.append(dict(rep=rep, model=mid, seed=seed, vocab=len({t for s in F.tokens for t in s}), **row99(yte, sv, st, scl)))
        if RUN_LSTM and rep in REPS_MAIN:
            key = f"split{seed}" if rep == "R1 full" else f"e4_{rep}_{seed}"
            res = get_lstm(key, F, C, seed)
            sv = lstm_score(res, C.tokens.tolist()); st = lstm_score(res, T.tokens.tolist())
            allrows.append(dict(rep=rep, model="LSTM-AE", seed=seed, vocab=len(res["vocab"]), **row99(yte, sv, st, scl)))
A = pd.DataFrame(allrows)

RESULTS["_e4_runs"] = A
first = A[A.seed == SEED] if SEED in E4_SEEDS else A[A.seed == E4_SEEDS[0]]

show("TABLE 15a  E4 " % first.seed.iloc[0],
     first.set_index(["model", "rep"])[["vocab","acc","prec","rec","f1","FP_N","FP_G","auc"]])

agg = A.groupby(["model", "rep"]).agg(vocab=("vocab","first"), runs=("seed","size"), auc_m=("auc","mean"), auc_sd=("auc","std"),
        f1_m=("f1","mean"), f1_sd=("f1","std"), FP_N=("FP_N","mean"), FP_G_m=("FP_G","mean"), FP_G_sd=("FP_G","std"))

show("TABLE 15b  E4 mean and SD over %d seeds" % len(E4_SEEDS), agg)

In [ ]:
#  E5: class imbalance

isN = (te.scenario.str[0] == "N").values; isA = (te.y == 1).values
rows = []
PCT = {m: 99 for m in S}; PCT["IF-eng"] = 95

for mid in ORDER:
    sv, st = S[mid]; thr = np.percentile(sv, PCT[mid])
    tpr = float((st[isA] > thr).mean()); k = int((st[isN] > thr).sum()); fpr = k / isN.sum()
    lo, hi = binomtest(k, int(isN.sum())).proportion_ci(method="exact")

    r = dict(model=mid, pct=PCT[mid], TPR=tpr, FPR=fpr, FPR_95ci=f"[{lo:.3f}, {hi:.3f}]", false_alarms_per_1000_normal=1000 * fpr)

    for pi in (0.05, 0.01, 0.001):
        r[f"precision@{pi:.1%}"] = tpr * pi / (tpr * pi + fpr * (1 - pi)) if tpr * pi + fpr * (1 - pi) > 0 else np.nan

    rows.append(r)

show("TABLE 16a  expected precision at 5%, 1%,)",
     pd.DataFrame(rows).set_index("model"))

In [ ]:
#Class imbalance with real files...
E5_FILES = {"E5 5% (570N+30A)": "e5_5pct.csv", "E5 1% (2976N+30A)": "e5_1pct.csv"}

present = {k: v for k, v in E5_FILES.items() if os.path.exists(v)}
if not present:
    print("E5 files not found --> table 16b skipping...", list(E5_FILES.values()))
else:

    e5 = {k: tx_table(pd.read_csv(v)) for k, v in present.items()}
    trn_seqs = {tuple(s) for s in tr.tokens}
    chk = []

    for k, d_ in e5.items():

        chk.append(dict(dataset=k, normal=int((d_.y == 0).sum()), anomalous=int(d_.y.sum()),
                        prevalence=d_.y.mean(), generation_seeds=sorted(map(int, d_.gseed.unique())),
                        seeds_shared_with_train=sorted(set(map(int, d_.gseed.unique())) & set(map(int, tr.gseed.unique()))),
                        scenarios_with_G=bool((d_.scenario.str[0] == "G").any()),
                        normal_token_sequences_also_in_train=float(np.mean([tuple(s) in trn_seqs for s in d_[d_.y == 0].tokens]))))

    show("TABLE 16b-i  E5 file checks ",
         pd.DataFrame(chk).set_index("dataset"))
    if len(e5) == 2:
        a, b = list(e5.values()); print("transaction ids :", len(set(a.transaction_id) & set(b.transaction_id)),

              "| anomalous transaction ids shared:", len(set(a[a.y == 1].transaction_id) & set(b[b.y == 1].transaction_id)))
    rows = []
    for k, d_ in e5.items():
        for mid in ORDER:
            sv = S[mid][0]
            if mid == "LSTM-AE": s_ = lstm_score(get_lstm("split42", FIT, CAL, SEED), d_.tokens.tolist())
            else:                _, (s_,) = sk_scores(mid, FIT, CAL, [d_])
            m, _ = conf(d_.y.values, s_, np.percentile(sv, PCT[mid]))
            rows.append(dict(dataset=k, model=mid, pct=PCT[mid], TP=m["tp"], FP=m["fp"], FN=m["fn"], TN=m["tn"], acc=m["acc"],
                             prec=m["prec"], rec=m["rec"], f1=m["f1"], fpr=m["fpr"]))
    show("TABLE 16b-ii  E5", pd.DataFrame(rows).set_index(["dataset","model"]), 4)

## Second independent test set (generation seed 33): Tables 17a–17d

Scores the **already fitted** detectors on the new test file. Nothing is refitted and no threshold is recalibrated:
thresholds are still the 95th/99th percentile of the normal calibration scores from the default split. Put the new
file next to the others (upload it in the first cell) and set `TEST2_FILE` below.

In [ ]:
#TABLES 17a-17d: second independent test set

TEST2_FILE = globals().get("TEST2_FILE", "test_002_v1.csv")

if not os.path.exists(TEST2_FILE):
    print("Second test file not found -> Tables 17a-17d skipped....", TEST2_FILE)
else:

    te2 = tx_table(pd.read_csv(TEST2_FILE)); y2 = te2.y.values
    seeds2 = set(map(int, te2.gseed.unique()))

    chk = dict(transactions=len(te2), normal=int((y2 == 0).sum()), anomalous=int(y2.sum()),
               generation_seeds=sorted(seeds2),
               seeds_shared_with_train=sorted(seeds2 & set(map(int, tr.gseed.unique()))),
               seeds_shared_with_test1=sorted(seeds2 & set(map(int, te.gseed.unique()))),
               transaction_ids_shared_with_test1=len(set(te2.transaction_id) & set(te.transaction_id)),
               same_scenario_counts_as_test1=bool(te2.scenario.value_counts().sort_index().equals(te.scenario.value_counts().sort_index())))

    show("TABLE 17a-0  second test set",
         pd.DataFrame([chk]).T.rename(columns={0: "value"}))
    assert not chk["seeds_shared_with_train"], "second test set reuses a training seed"

    S2 = {}
    for mid in MODELS:
        _, (st,) = sk_scores(mid, FIT, CAL, [te2]); S2[mid] = (S[mid][0], st)
    if RUN_LSTM:

        S2["LSTM-AE"] = (S["LSTM-AE"][0], lstm_score(get_lstm("split42", FIT, CAL, SEED), te2.tokens.tolist()))

    isN1, isG1 = (te.scenario.str[0] == "N").values, (te.scenario.str[0] == "G").values
    isN2, isG2 = (te2.scenario.str[0] == "N").values, (te2.scenario.str[0] == "G").values

    def fp_at(sv, st, mask, pct): return int((st[mask] > np.percentile(sv, pct)).sum())

    rows = []

    for mid in ORDER:
        sv, s1 = S[mid]; _, s2 = S2[mid]
        a1, a2 = roc_auc_score(yte, s1), roc_auc_score(y2, s2)
        lo, hi = boot_ci(y2, s2)
        yp, sp = np.concatenate([yte, y2]), np.concatenate([s1, s2])
        plo, phi = boot_ci(yp, sp)
        m1, _ = conf(yte, s1, np.percentile(sv, 99)); m2, _ = conf(y2, s2, np.percentile(sv, 99))

        rows.append(dict(model=mid, auc_test1=a1, auc_test2=a2, auc2_95ci=f"[{lo:.3f}, {hi:.3f}]",
                         auc_pooled=roc_auc_score(yp, sp), pooled_95ci=f"[{plo:.3f}, {phi:.3f}]",
                         ap_test2=average_precision_score(y2, s2), f1_99_test1=m1["f1"], f1_99_test2=m2["f1"],
                         FP_G_99_test1=fp_at(sv, s1, isG1, 99), FP_G_99_test2=fp_at(sv, s2, isG2, 99),
                         FP_N_99_test1=fp_at(sv, s1, isN1, 99), FP_N_99_test2=fp_at(sv, s2, isN2, 99)))

    T17 = pd.DataFrame(rows).set_index("model")

    T17["rank_test1"] = T17.auc_test1.rank(ascending=False).astype(int)
    T17["rank_test2"] = T17.auc_test2.rank(ascending=False).astype(int)
    show("TABLE 17a", T17)
    rho = spearmanr(T17.auc_test1, T17.auc_test2).correlation
    print(f"\nSpearman correlation of the detector ranking by ROC-AUC, test 1 vs test 2: {rho:.3f}")

    print("Ranking identical on both sets:", bool((T17.rank_test1 == T17.rank_test2).all()))

    def flagged2(pct):
        out = {mid: pd.Series((S2[mid][1] > np.percentile(S2[mid][0], pct)).astype(int)).groupby(te2.scenario.values).sum() for mid in ORDER}
        SC = sorted(te2.scenario.unique()); T = pd.DataFrame(out).T[SC]
        T.loc["(n per scenario)"] = pd.Series(1, index=range(len(te2))).groupby(te2.scenario.values).sum()[SC]
        return T
    show("TABLE 17b-95", flagged2(95), 0)
    show("TABLE 17b-99", flagged2(99), 0)

    def paired_on(Sd, y, a, b, seed=1):
        (sva, sta), (svb, stb) = Sd[a], Sd[b]; rng = np.random.default_rng(seed); d = []
        for _ in range(N_BOOT):
            i = boot_idx(y, rng); d.append(roc_auc_score(y[i], sta[i]) - roc_auc_score(y[i], stb[i]))
        lo, hi = np.percentile(d, [2.5, 97.5])
        ca, cb = ((sta > np.percentile(sva, 99)) == y), ((stb > np.percentile(svb, 99)) == y)
        b01, b10 = int((ca & ~cb).sum()), int((~ca & cb).sum())
        p = binomtest(b01, b01 + b10, 0.5).pvalue if b01 + b10 else 1.0
        return roc_auc_score(y, sta) - roc_auc_score(y, stb), lo, hi, b01, b10, p
    rows = []
    for a, b in pairs:
        d1 = roc_auc_score(yte, S[a][1]) - roc_auc_score(yte, S[b][1])
        d2, lo, hi, b01, b10, p = paired_on(S2, y2, a, b)
        rows.append(dict(A=a, B=b, dAUC_test1=d1, dAUC_test2=d2, ci95_test2=f"[{lo:.3f}, {hi:.3f}]",
                         excludes_0=bool(lo > 0 or hi < 0), same_direction=bool(np.sign(d1) == np.sign(d2)),
                         A_right_B_wrong=b01, B_right_A_wrong=b10, mcnemar_p=p))
    show("TABLE 17c",
         pd.DataFrame(rows).set_index(["A", "B"]))

    def run_split_t(seed, ids, tdf):
        fi, ci = split(seed); F, C = tr.iloc[fi], tr.iloc[ci]; y = tdf.y.values; out = {}
        for mid in ids:
            if mid == "LSTM-AE": sv, (st,) = lstm_scores(f"split{seed}", F, C, [tdf], seed)
            else:                sv, (st,) = sk_scores(mid, F, C, [tdf], seed=seed)
            r = summarize(y, sv, st); r["FP_G@99"] = fp_at(sv, st, (tdf.scenario.str[0] == "G").values, 99); out[mid] = r
        return out
    runs2 = [run_split_t(s, ids4, te2) for s in SPLIT_SEEDS]
    rows = []
    for mid in ids4:
        d = pd.DataFrame([r[mid] for r in runs2]); row = {"model": mid, "runs": len(d)}
        for c in ["auc", "ap", "f1@99", "fpr@99", "FP_G@99"]:
            row[c] = f"{d[c].mean():.3f} ± {d[c].std(ddof=1):.3f}"
        rows.append(row)
    show("TABLE 17d  second test set: mean ± SD over %d calibration splits" % len(SPLIT_SEEDS),
         pd.DataFrame(rows).set_index("model"))

In [ ]:
# Figures 3–8
# Model evaluation and comparison plots

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import roc_curve, roc_auc_score

os.makedirs("figures", exist_ok=True)

plt.rcParams.update({
    "font.size": 10,
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False
})

model_names = {
    "LSTM-AE": "LSTM autoencoder",
    "IF-eng": "Isolation Forest (engineered)",
    "LOF-count": "LOF, token counts",
    "LOF-pair": "LOF, token pairs",
    "OCSVM-pair": "One-class SVM, token pairs"
}

model_colors = {
    "LSTM-AE": "#0072B2",
    "IF-eng": "#E69F00",
    "LOF-count": "#56B4E9",
    "LOF-pair": "#009E73",
    "OCSVM-pair": "#CC79A7"
}

models = [name for name in model_names if name in S]
second_test_available = "S2" in globals()


def save_plot(filename, adjust_layout=True):
    if adjust_layout:
        plt.tight_layout()

    plt.savefig(
        os.path.join("figures", filename + ".png"),
        dpi=300
    )
    plt.show()


# Figure 3. ROC curves for the first test set

fig, ax = plt.subplots(figsize=(5, 4.5))

for model in models:
    calibration_scores, test_scores = S[model]

    fpr, tpr, _ = roc_curve(yte, test_scores)
    auc = roc_auc_score(yte, test_scores)

    ax.plot(
        fpr, tpr,
        color=model_colors[model],
        linewidth=1.6,
        label=f"{model_names[model]} (AUC {auc:.3f})"
    )

ax.plot([0, 1], [0, 1], "k:", linewidth=0.8)

ax.set_xlabel("False-positive rate")
ax.set_ylabel("True-positive rate")
ax.legend(loc="lower right", frameon=False, fontsize=8)

save_plot("fig3_roc")


# Figure 4. Percentage of transactions flagged in each scenario

scenarios = sorted(te.scenario.unique())


def get_flag_rates(scores, test_data):
    rates = np.zeros((len(models), len(scenarios)))

    for i, model in enumerate(models):
        calibration_scores, test_scores = scores[model]
        threshold = np.percentile(calibration_scores, 99)

        flagged = (test_scores > threshold).astype(int)

        scenario_rates = (
            pd.Series(flagged)
            .groupby(test_data.scenario.values)
            .mean()
        )

        rates[i] = scenario_rates[scenarios].values

    return rates


test_sets = [("Test set 1 (seed 30)", S, te)]

if second_test_available:
    test_sets.append(("Test set 2 (seed 33)", S2, te2))

fig, axes = plt.subplots(
    1, len(test_sets),
    figsize=(5.6 * len(test_sets), 3.2),
    squeeze=False,
    layout="constrained"
)

for index, (title, scores, test_data) in enumerate(test_sets):
    ax = axes[0, index]
    rates = get_flag_rates(scores, test_data)

    image = ax.imshow(
        rates,
        vmin=0,
        vmax=1,
        cmap="Blues",
        aspect="auto"
    )

    ax.set_xticks(range(len(scenarios)))
    ax.set_xticklabels(scenarios, rotation=90)

    ax.set_yticks(range(len(models)))

    if index == 0:
        ax.set_yticklabels([model_names[m] for m in models])
    else:
        ax.set_yticklabels([])

    for row in range(rates.shape[0]):
        for col in range(rates.shape[1]):
            value = rates[row, col]

            if value in (0, 1):
                label = str(int(value * 100))
            else:
                label = f"{value:.0%}"

            ax.text(
                col, row, label,
                ha="center",
                va="center",
                fontsize=6,
                color="white" if value > 0.6 else "black"
            )

    ax.set_title(title, fontsize=10)

fig.colorbar(
    image,
    ax=axes[0].tolist(),
    fraction=0.03,
    pad=0.02,
    label="share flagged"
)

save_plot("fig4_scenario_flag_rates", adjust_layout=False)


# Figure 5. Anomaly score distributions

selected_models = [
    model for model in ("LSTM-AE", "LOF-pair")
    if model in S
]

if not selected_models:
    selected_models = ["LOF-pair"]

groups = {
    "N (familiar normal)": (te.scenario.str[0] == "N").values,
    "G (unseen legitimate)": (te.scenario.str[0] == "G").values,
    "A (anomalous)": (te.scenario.str[0] == "A").values
}

group_colors = ["#009E73", "#E69F00", "#D55E00"]

fig, axes = plt.subplots(
    1, len(selected_models),
    figsize=(4.6 * len(selected_models), 3.4),
    squeeze=False
)

for ax, model in zip(axes[0], selected_models):
    calibration_scores, test_scores = S[model]

    upper_limit = np.percentile(
        np.r_[test_scores, calibration_scores], 99.5
    )

    bins = np.linspace(
        min(test_scores.min(), calibration_scores.min()),
        upper_limit,
        40
    )

    for (group_name, mask), color in zip(groups.items(), group_colors):
        values = np.clip(test_scores[mask], None, upper_limit)

        ax.hist(
            values,
            bins=bins,
            alpha=0.55,
            color=color,
            label=group_name
        )

    threshold = np.percentile(calibration_scores, 99)

    ax.axvline(
        threshold,
        color="black",
        linestyle="--",
        linewidth=1,
        label="99th-percentile threshold"
    )

    ax.set_title(model_names[model])
    ax.set_xlabel("anomaly score (clipped at the 99.5th percentile)")
    ax.set_ylabel("transactions")

axes[0][0].legend(frameon=False, fontsize=7)

save_plot("fig5_score_distributions")


# Figure 6. Expected precision at different anomaly prevalences

normal_mask = (te.scenario.str[0] == "N").values
anomaly_mask = (te.y == 1).values

prevalence = np.logspace(-3, np.log10(0.2), 60)

fig, ax = plt.subplots(figsize=(5, 4))

for model in models:
    calibration_scores, test_scores = S[model]

    percentile = 95 if model == "IF-eng" else 99
    threshold = np.percentile(calibration_scores, percentile)

    tpr = float((test_scores[anomaly_mask] > threshold).mean())
    fpr = float((test_scores[normal_mask] > threshold).mean())

    precision = []

    for p in prevalence:
        numerator = tpr * p
        denominator = numerator + fpr * (1 - p)

        if denominator > 0:
            precision.append(numerator / denominator)
        else:
            precision.append(np.nan)

    ax.plot(
        prevalence,
        precision,
        color=model_colors[model],
        linewidth=1.6,
        label=f"{model_names[model]} (TPR {tpr:.2f}, FPR {fpr:.3f})"
    )

ax.set_xscale("log")
ax.set_ylim(0, 1.02)

ax.set_xlabel("anomaly prevalence")
ax.set_ylabel("expected precision")
ax.legend(frameon=False, fontsize=7)

save_plot("fig6_precision_vs_prevalence")


# Figure 7. LOF sensitivity and training-set size

fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))

# (a) Effect of neighbourhood size

k_values = [5, 10, 20, 50, 100]
auc_values = []
f1_values = []

for k in k_values:
    calibration_scores, (test_scores,) = sk_scores(
        "LOF-pair", FIT, CAL, [te], k=k
    )

    auc_values.append(roc_auc_score(yte, test_scores))

    metrics = summarize(yte, calibration_scores, test_scores)
    f1_values.append(metrics["f1@99"])

axes[0].plot(
    k_values, auc_values,
    "o-",
    color="#009E73",
    label="ROC-AUC"
)

axes[0].plot(
    k_values, f1_values,
    "s--",
    color="#D55E00",
    label="F1 at 99th percentile"
)

axes[0].set_xscale("log")
axes[0].set_xticks(k_values)
axes[0].set_xticklabels(k_values)
axes[0].set_xlabel("LOF neighbourhood size k")
axes[0].set_ylim(0, 1.02)
axes[0].set_title("(a) sensitivity to k")
axes[0].legend(frameon=False, fontsize=8)


# (b) Effect of training-set size

table_key = [
    key for key in RESULTS
    if key.startswith("TABLE 6a")
][0]

training_results = RESULTS[table_key]

for model in training_results.index:
    if model in model_colors:
        axes[1].plot(
            training_results.columns,
            training_results.loc[model],
            "o-",
            color=model_colors[model],
            label=model_names[model]
        )

axes[1].set_xlabel("normal training transactions")
axes[1].set_ylabel("ROC-AUC")
axes[1].set_title("(b) training-set size")
axes[1].legend(frameon=False, fontsize=7)

save_plot("fig7_lof_k_and_training_size")


# Figure 8. Comparison between the two test sets

if second_test_available:
    fig, ax = plt.subplots(figsize=(6.2, 3.6))

    comparison_models = [m for m in ids4 if m in model_names]

    x = np.arange(len(comparison_models))
    bar_width = 0.38

    comparisons = [
        ("Test set 1 (seed 30)", runs, "#7f7f7f"),
        ("Test set 2 (seed 33)", runs2, "#0072B2")
    ]

    for offset, (label, results, color) in zip(
        [-bar_width / 2, bar_width / 2],
        comparisons
    ):
        auc_results = [
            [run[model]["auc"] for run in results]
            for model in comparison_models
        ]

        mean_auc = [np.mean(values) for values in auc_results]
        std_auc = [
            np.std(values, ddof=1) for values in auc_results
        ]

        ax.bar(
            x + offset,
            mean_auc,
            bar_width,
            yerr=std_auc,
            capsize=2,
            color=color,
            label=label
        )

    labels = [
        model_names[m].replace(", ", ",\n").replace(" (", "\n(")
        for m in comparison_models
    ]

    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=7)
    ax.set_ylabel("ROC-AUC")
    ax.set_ylim(0.5, 1.02)
    ax.legend(frameon=False, fontsize=8)

    save_plot("fig8_two_test_sets")

else:
    print("Figure 8 skipped: no second test set loaded.")


In [ ]:
N_RUNS = 3
rows = []

# --- rerun with the same seed and data
for mid in MODELS:
    sc, pr = [], []
    for _ in range(N_RUNS):
        sv, (st,) = sk_scores(mid, FIT, CAL, [te])
        pct = 95 if mid == "IF-eng" else 99
        sc.append(st); pr.append(st > np.percentile(sv, pct))
    rows.append(dict(model=mid,
                     scores_identical=all(np.array_equal(sc[0], s) for s in sc[1:]),
                     max_abs_score_diff=max(np.abs(sc[0] - s).max() for s in sc[1:]),
                     predictions_identical=all(np.array_equal(pr[0], p) for p in pr[1:]),
                     roc_auc=[round(roc_auc_score(yte, s), 4) for s in sc][0]))

# --- LSTM: rerun with seed 42
if RUN_LSTM:
    sc, pr, aucs = [], [], []
    for _ in range(N_RUNS):
        res = lstm_fit(FIT.tokens.tolist(), CAL.tokens.tolist(), 42)
        s = lstm_score(res, te.tokens.tolist())
        thr = np.percentile(lstm_score(res, CAL.tokens.tolist()), 99)
        sc.append(s); pr.append(s > thr); aucs.append(roc_auc_score(yte, s))
    rows.append(dict(model="LSTM-AE",
                     scores_identical=all(np.array_equal(sc[0], s) for s in sc[1:]),
                     max_abs_score_diff=max(np.abs(sc[0] - s).max() for s in sc[1:]),
                     predictions_identical=all(np.array_equal(pr[0], p) for p in pr[1:]),
                     roc_auc=round(aucs[0], 4)))
    print("LSTM ROC-AUC per run:", [round(a, 4) for a in aucs])

show("TABLE E6  reproducibility: three reruns with seed 42 on identical data", pd.DataFrame(rows).set_index("model"), 6)


In [ ]:
# E6 --- Dataset regeneration
a, b = clean(pd.read_csv("test_001_v1.csv")), clean(pd.read_csv("test_003_v1.csv"))
print("test datasets identical:", a.equals(b), "| rows:", len(a), len(b))

In [ ]:
# E6
a = a.reset_index(drop=True); b = b.reset_index(drop=True)
print("columns that differ:", [c for c in a.columns if not a[c].equals(b[c])])

ta, tb = tx_table(a), tx_table(b)
m = ta.merge(tb, on="transaction_id", suffixes=("_a", "_b"))
print("same transaction ids:", set(ta.transaction_id) == set(tb.transaction_id), "| matched:", len(m))
print("identical token sequence:", (m.tokens_a.map(tuple) == m.tokens_b.map(tuple)).mean())
print("identical labels:", (m.y_a == m.y_b).mean(), "| identical scenarios:", (m.scenario_a == m.scenario_b).mean())

ta, tb = tx_table(a), tx_table(b)
print("transactions:", len(ta), len(tb))
print("identical token sequences:", np.mean([tuple(x) == tuple(y) for x, y in zip(ta.tokens, tb.tokens)]))
print("identical labels:", (ta.y.values == tb.y.values).mean(), "| identical scenarios:", (ta.scenario.values == tb.scenario.values).mean())

In [ ]:
vocab_fit = {t for seq in FIT.tokens for t in seq}

unseen = {
    t
    for seq in te.tokens
    for t in seq
    if t not in vocab_fit
}

print("Total distinct OOV tokens:", len(unseen))
print("\nOOV token identities:")

for i, token in enumerate(sorted(unseen), 1):
    print(f"{i}. {token}")

In [ ]:
# ------------------------------------------------------------------ save everything.....
import re, zipfile
os.makedirs("tables", exist_ok=True)
for name, df in RESULTS.items():
    df.to_csv("tables/" + re.sub(r"[^A-Za-z0-9]+", "_", name.split("  ")[0]).strip("_") + ".csv")
with zipfile.ZipFile("paper_tables.zip", "w") as z:
    for f in sorted(os.listdir("tables")): z.write("tables/" + f)
    if os.path.isdir("figures"):
        for f in sorted(os.listdir("figures")): z.write("figures/" + f)
print("saved", len(RESULTS), "tables")
try:
    files.download("paper_tables.zip")
except Exception as e:
    print("download not available here:", e)